## 1.4 章节实践

通过本章的系统学习，我们已经掌握了 GFSK 调制与解调的完整原理，从数字调制的概念出发，理解了 NRZ 映射、高斯滤波、相位积分和复指数映射四步调制流程，以及频率鉴别器非相干解调的实现机制，并通过 BER-SNR 扫描验证了仿真链路的正确性。为了巩固所学知识，现提供以下实践练习：

**从零实现 GFSK 调制与解调**，不调用 `GFSKModulator` 和 `GFSKDemodulator`，仅使用 NumPy 手写完整的 GFSK 链路。

**相关算法：**

调制：比特 → NRZ 映射 → 上采样 → 高斯滤波 → 相位积分 → 复指数映射 → IQ 信号

解调：IQ 信号 → 相邻采样点相位差 → 按符号分组累加 → 正负判决 → 比特

**要求：**

1. 补全 `my_gfsk_modulate` 中相位积分和复指数映射两步
2. 补全 `my_gfsk_demodulate` 中按符号累积判决的循环
3. 在 SNR=8 dB 下对比手写实现与库函数的结果，验证一致性

请开始你的实践，体验从理解到创造的完整开发过程。"]

In [ ]:
%%writefile my_gfsk.py
import sys
sys.path.insert(0, "../src")
import numpy as np

def _gaussian_filter(bt, span, sps):
    N = 2 * span * sps + 1
    t = np.arange(-(N-1)/2, (N-1)/2+1) / sps
    alpha = np.sqrt(np.log(2)/2) / bt
    h = np.sqrt(2*np.pi) / alpha * np.exp(-2 * (np.pi*t/alpha)**2)
    return h / np.sum(h)

def my_gfsk_modulate(bits, sps=8, mod_index=0.5, bt=0.5, span=3):
    # 1. NRZ 映射
    nrz = 2.0 * np.asarray(bits, dtype=float) - 1.0
    # 2. 上采样
    upsampled = np.repeat(nrz, sps)
    # 3. 高斯滤波
    gauss = _gaussian_filter(bt, span, sps)
    filtered = np.convolve(upsampled, gauss, mode='same')
    # 4. 相位积分 —— : 补全
    freq_dev = mod_index * np.pi / sps
    #待补充
    phase = 
    # 5. 复指数映射 —— : 补全
    #待补充
    signal = 
    return signal

def my_gfsk_demodulate(signal, sps=8):
    # 1. 频率鉴别: 相邻采样点相位差
    phase_diff = np.angle(signal[1:] * np.conj(signal[:-1]))
    n_symbols = -(-len(phase_diff) // sps)
    bits = np.zeros(n_symbols, dtype=int)
    # 2. 按符号累积判決 —— : 补全循环体
    for i in range(n_symbols):
        #待补充

    return bits

# ---- 测试 ----
if __name__ == '__main__':
    from nearlink_sdr.phy.gfsk import GFSKModulator, GFSKDemodulator
    from nearlink_sdr.phy.channel import ChannelModel

    rng = np.random.default_rng(42)
    tx_bits = rng.integers(0, 2, 10000)

    tx_my = my_gfsk_modulate(tx_bits)
    ch = ChannelModel(snr_db=8.0)
    rx = ch.apply_awgn(tx_my)
    rx_my = my_gfsk_demodulate(rx)

    mod = GFSKModulator(sps=8)
    tx_lib = mod.modulate(tx_bits)
    ch2 = ChannelModel(snr_db=8.0)
    rx2 = ch2.apply_awgn(tx_lib)
    demod = GFSKDemodulator(sps=8)
    rx_lib = demod.demodulate(rx2)

    # 对比
    ber_my = np.mean(tx_bits[:len(rx_my)] != rx_my[:len(rx_my)])
    ber_lib = np.mean(tx_bits[:len(rx_lib)] != rx_lib[:len(rx_lib)])
    print(f'手写 BER: {ber_my:.6f}')
    print(f'库函数 BER: {ber_lib:.6f}')
    print(f'一致: {abs(ber_my-ber_lib) < 0.01}')

执行以下命令进行编译并验证结果：


In [ ]:
!python my_gfsk.py


执行以下代码获取答案

完成实践后，可运行以下 cell 查看参考实现：

In [ ]:
!cat answer/my_gfsk_answer.py